# 什么是with上下文管理？
- with 是 Python 中用于上下文管理的语法糖，核心作用是自动管理资源（如文件、数据库连接、锁等），确保资源在使用后被正确释放，避免资源泄漏。
- with 能让你在 “进入某个代码块前自动执行准备工作”，“离开代码块后自动执行清理工作”，即使代码块中发生异常也不例外。

# 一、为什么需要with？
这里用 try-finally 确保文件被关闭，但代码冗余且容易忘记写 finally。

In [1]:
# 不推荐：手动管理资源，容易出错
f = open("../0-文件/test.txt", "r", encoding="utf-8")
try:
    content = f.read()
    print(content)
finally:
    f.close()  # 必须手动关闭，否则文件资源泄漏

第一行：这是一个测试文件-test.txt \n
第二行：人生苦短，我用Python


使用with后

In [2]:
# 推荐：自动管理资源
with open("../0-文件/test.txt", "r", encoding="utf-8") as f:
    content = f.read()
    print(content)
# 离开 with 块后，文件自动关闭，无需手动调用 f.close()

第一行：这是一个测试文件-test.txt \n
第二行：人生苦短，我用Python


# 二、上下文管理器的核心协议
with 之所以能自动管理资源，是因为它依赖 Python 的上下文管理器协议：任何实现了 \_\_enter__() 和 \_\_exit__() 方法的对象，都是上下文管理器。
1. \_\_enter__()方法
    - 进入with代码块前自动执行。
    - 返回值会赋给as后面的变量（如 with ... as f 中的f）。
2. \_\_exit__()方法
    - 离开with代码块后自动执行（无论是否发生异常）。
    - 按3个参数：
        - exc_type：异常类型（无异常时为 None）。
        - exc_val：异常对象（无异常时为 None）。
        - exc_tb：异常追踪信息（无异常时为 None）。
    - 若返回True，则吞掉异常（不继续向外传播）；返回False或None则继续传播异常。

# 三、自定义上下文管理器
我们可以通过实现 \_\_enter__ 和 \_\_exit__ 方法，自己写一个上下文管理器。比如模拟一个 “数据库连接” 的资源管理：

In [3]:
class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name

    # 进入 with 块时执行：模拟连接数据库
    def __enter__(self):
        print(f"正在连接数据库：{self.db_name}")
        self.connection = f"已连接到 {self.db_name}"  # 模拟连接对象
        return self.connection  # 返回给 as 后的变量

    # 离开 with 块时执行：模拟关闭连接
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"正在关闭数据库连接：{self.db_name}")
        # 若有异常，可在这里处理
        if exc_type:
            print(f"捕获到异常：{exc_type} - {exc_val}")
        # 返回 False 表示不吞掉异常，继续向外传播
        return False

# 使用自定义上下文管理器
with DatabaseConnection("mydb") as conn:
    print(conn)  # 输出：已连接到 mydb
    # 模拟操作数据库
    # 若这里触发异常，__exit__ 仍会执行
    # raise ValueError("操作出错")  # 可取消注释测试异常情况

正在连接数据库：mydb
已连接到 mydb
正在关闭数据库连接：mydb


# 四、更简单的方式：contextlib 模块
对于简单的上下文管理，无需写类，可用 contextlib.contextmanager 装饰器 + 生成器函数实现：

In [4]:
from contextlib import contextmanager

@contextmanager
def database_connection(db_name):
    # __enter__ 的逻辑：进入 with 块前执行
    print(f"正在连接数据库：{db_name}")
    conn = f"已连接到 {db_name}"
    try:
        yield conn  # 返回给 as 后的变量，暂停执行，等待 with 块结束
    finally:
        # __exit__ 的逻辑：离开 with 块后执行
        print(f"正在关闭数据库连接：{db_name}")

# 使用
with database_connection("mydb") as conn:
    print(conn)

正在连接数据库：mydb
已连接到 mydb
正在关闭数据库连接：mydb


效果和自定义类一样，但代码更简洁。

# 常见应用场景
1. 文件操作：自动关闭文件（最常用）。
2. 数据库连接：自动连接 / 关闭。
3. 线程锁：自动获取 / 释放锁（如 threading.Lock）。
4. 临时目录 / 文件：自动创建 / 删除（如 tempfile.TemporaryDirectory）。